# Project: Eigenfaces 👤
The classic 1991 face-recognition idea (Turk & Pentland): treat every 64×64 face as a point in 4,096-dimensional space and find its principal components, the **eigenfaces**. A face becomes ~50 numbers instead of 4,096. Reconstruct faces from a few components, then recognise people with nearest neighbours in eigenface space.

Topic: [[PCA]], [[Dimensionality Reduction]] · guide note: [[Project - Eigenfaces]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
faces = fetch_olivetti_faces(data_home="data")                   # 400 images of 40 people (AT&T Laboratories Cambridge), ~4 MB
X, y = faces.data, faces.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(X.shape, "pixels per face:", X.shape[1])

## 1. PCA via SVD
Centre the data, take the SVD $X - \bar x = U\Sigma V^\top$; the rows of $V^\top$ are the principal directions. Return `(mean, first k components, explained variances)` where variance $= \sigma^2/(n-1)$ for **all** singular values.

In [ ]:
def pca(X, k):
    # TODO 1: mean, top-k right singular vectors, all explained variances
    raise NotImplementedError  # TODO 1

## 2. Project and reconstruct
Coordinates $Z = (X-\bar x)V_k^\top$, reconstruction $\hat X = ZV_k + \bar x$.

In [ ]:
def reconstruct(X, mean, components):
    # TODO 2: project onto the components and map back
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_pca():
    from sklearn.decomposition import PCA
    mean, comp, var = pca(X_tr, 10); sk = PCA(10).fit(X_tr)
    same_dirs = np.allclose(np.abs(np.sum(comp * sk.components_, 1)), 1, atol=1e-4)
    full = reconstruct(X_tr[:5], *pca(X_tr, len(X_tr))[:2])
    return same_dirs and np.allclose(var[:10], sk.explained_variance_, rtol=1e-4) and np.allclose(full, X_tr[:5], atol=1e-4)
check("PCA", _t_pca, "Components must match sklearn's PCA (up to sign); with all components the reconstruction is exact.")
if ready("PCA"):
    mean, comp, var = pca(X_tr, 150)
    fig, ax = plt.subplots(2, 8, figsize=(13, 3.6))
    ax[0, 0].imshow(mean.reshape(64, 64), cmap="gray"); ax[0, 0].set_title("mean face", fontsize=8)
    for j in range(1, 8): ax[0, j].imshow(comp[j - 1].reshape(64, 64), cmap="gray"); ax[0, j].set_title(f"eigenface {j}", fontsize=8)
    for j, k in enumerate((1, 5, 10, 25, 50, 100, 150)): ax[1, j].imshow(reconstruct(X_te[:1], mean, comp[:k])[0].reshape(64, 64), cmap="gray"); ax[1, j].set_title(f"k = {k}", fontsize=8)
    ax[1, 7].imshow(X_te[0].reshape(64, 64), cmap="gray"); ax[1, 7].set_title("original", fontsize=8)
    for a in ax.ravel(): a.axis("off")
    plt.show()
    cum = np.cumsum(var) / var.sum(); k90 = int(np.searchsorted(cum, 0.9)) + 1; print(f"{k90} components explain 90 % of the variance (of {X.shape[1]} pixels)")
    accs = {}
    for k in (5, 20, 50, 150):
        Ztr = (X_tr - mean) @ comp[:k].T; Zte = (X_te - mean) @ comp[:k].T
        accs[k] = KNeighborsClassifier(1).fit(Ztr, y_tr).score(Zte, y_te); print(f"1-NN face recognition with {k:>3} eigenfaces: {accs[k]:.1%}")
    raw = KNeighborsClassifier(1).fit(X_tr, y_tr).score(X_te, y_te); print(f"1-NN on all 4096 raw pixels: {raw:.1%}")
    check("eigenfaces recognise", lambda: accs[50] >= raw - 0.02 and accs[50] > 0.85, "50 eigenfaces should recognise people about as well as 4096 raw pixels (> 85 %).")

<details><summary>▶ Solution</summary>

```python
def pca(X, k):
    mean = X.mean(0)
    U, s, Vt = np.linalg.svd(X - mean, full_matrices=False)
    return mean, Vt[:k], s ** 2 / (len(X) - 1)
```

```python
def reconstruct(X, mean, components):
    Z = (X - mean) @ components.T
    return Z @ components + mean
```
</details>

## Stretch goals
- Drop the first 3 eigenfaces (they mostly capture lighting) and re-test recognition.
- Morph between two people by interpolating their eigenface coordinates.
- Compare with a pretrained CNN embedding: [[Danish Mushroom Classifier]] shows how.